In [ ]:
import pandas as pd

data = pd.DataFrame({
    "Better law","Process less","Too bad"
})
print(data)

              0
0  Process less
1    Better law
2       Too bad


In [ ]:
reviews = []

inp = input("Enter your Reviews:")

while inp != 'stop':
    reviews.append(inp)
    inp = input("Enter your Reviews:")

data = pd.DataFrame(reviews, columns=['Review'])
display(data)

Enter your Reviews:It is a very good product
Enter your Reviews:It is not good product
Enter your Reviews:it is not bad
Enter your Reviews:it is simple and unique product
Enter your Reviews:stop


,Review
0,It is a very good product
1,It is not good product
2,it is not bad
3,it is simple and unique product


In [ ]:
# Install the transformers library if you haven't already
!pip install transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Define the name of a pre-trained model from Hugging Face Model Hub
# This model is fine-tuned for sentiment analysis (positive/negative)
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(f"Successfully loaded tokenizer and model: {model_name}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Successfully loaded tokenizer and model: distilbert-base-uncased-finetuned-sst-2-english


### Performing Sentiment Analysis on Reviews

Now that the model and tokenizer are loaded, we can use them to predict the sentiment of the reviews you entered. The model outputs logits, which we'll convert into probabilities and then determine the sentiment label (Positive or Negative).

In [ ]:
# Ensure 'data' DataFrame is available from previous steps
# If you've run the cell to collect reviews, 'data' should contain them.
if not data.empty:
    reviews_to_analyze = data['Review'].tolist()
else:
    print("The DataFrame 'data' is empty. Please enter some reviews first.")
    reviews_to_analyze = []

if reviews_to_analyze:
    # Tokenize the input reviews
    # padding=True ensures all sequences are padded to the same length
    # truncation=True ensures sequences longer than model's max input are truncated
    inputs = tokenizer(reviews_to_analyze, padding=True, truncation=True, return_tensors="pt")

    # Pass the inputs through the model
    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = model(**inputs)

    # Get the logits (raw scores) from the model output
    logits = outputs.logits

    # Apply softmax to convert logits to probabilities
    probabilities = torch.softmax(logits, dim=1)

    # Get the predicted class (0 for negative, 1 for positive for this specific model)
    predicted_class_ids = torch.argmax(probabilities, dim=1).numpy()

    # Map class IDs to sentiment labels
    sentiment_labels = ['Negative', 'Positive'] # Based on the model's training
    predicted_sentiments = [sentiment_labels[class_id] for class_id in predicted_class_ids]

    # Add sentiment to the DataFrame
    data['Sentiment'] = predicted_sentiments
    print("Reviews with predicted sentiments:")
    display(data)
else:
    print("No reviews were analyzed as the review list was empty.")

Reviews with predicted sentiments:


,Review,Sentiment
0,It is a very good product,Positive
1,It is not good product,Negative
2,it is not bad,Positive
3,it is simple and unique product,Positive


# **Audio Sentiment Analysis:-**

In [ ]:
!pip install openai-whisper gtts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 10.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=1f7c27331788baf9d5c3dbab50d936813e78d3736505675014c078a859f78648
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency con

In [ ]:
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.4.1 which is incompatible.


In [ ]:
from gtts import gTTS
import whisper
from transformers import pipeline

In [ ]:
text = "I had an issue with how the subscription was presented after significant amount was deducted from my account.I am happy to report that I was promptly sorted. This was top notch customer service for audio team! I recommend them"
raw_audio = gTTS(text=text,lang="en")
raw_audio.save("raw.mp3")

**Customer Audio Review**

In [ ]:
model = whisper.load_model("base")
text_result = model.transcribe("raw.mp3")
print(text_result)

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


{'text': ' This is a super product and unique product.', 'segments': [{'id': 0, 'seek': 0, 'start': 0.0, 'end': 2.8000000000000003, 'text': ' This is a super product and unique product.', 'tokens': [50364, 639, 307, 257, 1687, 1674, 293, 3845, 1674, 13, 50504], 'temperature': 0.0, 'avg_logprob': -0.3867141405741374, 'compression_ratio': 1.0, 'no_speech_prob': 0.005090463440865278}], 'language': 'en'}


In [ ]:
print(text_result['text'])

 This is a super product and unique product.


**Sentiment Analysis**

In [ ]:
analyzer = pipeline("sentiment-analysis")
analysis_result = analyzer(text_result['text'])

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

In [ ]:
print(analysis_result)

[{'label': 'POSITIVE', 'score': 0.9997901320457458}]


**Classification and Conversion**

In [ ]:
if analysis_result[0]['label']=='POSITIVE':
    audio_result = gTTS(text=f"User gives {analysis_result[0]['label']} with a score of {int((analysis_result[0]['score'])*100)} percent",lang="en").save("result.mp3")
    from IPython.display import display,Audio
    display(Audio("result.mp3"))
else:
    print("User gives Negative review")

In [ ]:
from gtts import gTTS
import whisper
from transformers import pipeline

input_review = input("Enter the user review:")
# text = "I had an issue with how the subscription was presented after significant amount was deducted from my account.I am happy to report that I was promptly sorted. This was top notch customer service for audio team! I recommend them"
raw_audio = gTTS(text=input_review,lang="en")
raw_audio.save("raw.mp3")
model = whisper.load_model("base")
text_result = model.transcribe("raw.mp3")
analyzer = pipeline("sentiment-analysis")
analysis_result = analyzer(text_result['text'])
if analysis_result[0]['label']=='POSITIVE':
    audio_result = gTTS(text=f"User gives {analysis_result[0]['label']} with a score of {int((analysis_result[0]['score'])*100)} percent",lang="en").save("result.mp3")
    from IPython.display import display,Audio
    display(Audio("result.mp3"))
else:
    print("User gives Negative review")

Enter the user review:I am happy to report that I was promptly sorted. This was top notch customer service for audio team! I recommend them


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

**Amazon review Sentiment Analysis**

In [ ]:
!pip install pandas

In [1]:
!pip install playwright
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 14.8 MB/s eta 0:00:00
175.4 MiB [] 0% 361.4s175.4 MiB [] 0% 73.2s175.4 MiB [] 0% 41.3s175.4 MiB [] 0% 27.6s175.4 MiB [] 0% 24.7s175.4 MiB [] 0% 16.7s175.4 MiB [] 1% 10.1s175.4 MiB [] 1% 9.6s175.4 MiB [] 1% 7.4s175.4 MiB [] 2% 8.5s175.4 MiB [] 2% 9.1s175.4 MiB [] 2% 9.3s175.4 MiB [] 2% 9.9s175.4 MiB [] 3% 9.2s175.4 MiB [] 3% 8.3s175.4 MiB [] 4% 7.7s175.4 MiB [] 4% 7.1s175.4 MiB [] 5% 6.7s175.4 MiB [] 5% 6.2s175.4 MiB [] 6% 5.9s175.4 MiB [] 6% 5.7s175.4 MiB [] 7% 6.1s175.4 MiB [] 7% 5.8s175.4 MiB [] 7% 6.0s175.4 MiB [] 8% 6.2s175.4 MiB [] 8% 6.3s175.4 MiB [] 9% 5.7s175.4 MiB [] 10% 5.4s175.4 MiB [] 11% 5.1s175.4 MiB [] 11% 4.9s175.4 MiB [] 12% 4.8s175.4 MiB [] 12% 4.7s175.4 MiB [] 13% 4.4s175.4 MiB [] 14% 4.3s175.4 MiB [] 14% 4.2s175.4 MiB [] 15% 4.2s175.4 MiB [] 15% 4.1s175.4 MiB [] 16% 4.0s175.4 MiB [] 17% 3.8s175.4 MiB [] 18% 3.7s175.4 MiB [] 19% 3.6s175.4 MiB [] 19% 3.5s175.4 MiB [] 20% 3.5s175.4 MiB [] 20% 3.4s175.4 MiB [] 21%

In [6]:
!playwright install-deps

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,973 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,298 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 P

In [7]:
import json
from playwright.async_api import async_playwright

async def scrape_quotes_async():
    TARGET_URL = "https://quotes.toscrape.com/"

    print(f"Launching headless browser to scrape: {TARGET_URL}...")

    # Notice the 'async with' instead of just 'with'
    async with async_playwright() as p:
        # We must 'await' browser launching and page creation
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Navigate to the target website
        await page.goto(TARGET_URL)

        # Wait until the main content is loaded
        await page.wait_for_selector(".quote")

        # Grab all the quote blocks on the page
        quote_elements = await page.locator(".quote").all()
        scraped_quotes = []

        print(f"Found {len(quote_elements)} quotes. Extracting data...")

        for element in quote_elements:
            # We must await text extractions in the async API
            text = await element.locator(".text").text_content()
            author = await element.locator(".author").text_content()
            tags = await element.locator(".tag").all_text_contents()

            clean_text = text.strip().replace('“', '').replace('”', '')

            scraped_quotes.append({
                "quote": clean_text,
                "author": author.strip(),
                "tags": tags
            })

        await browser.close()
        return scraped_quotes

# In Jupyter/Colab, you can await the coroutine directly in the cell execution!
data = await scrape_quotes_async()

print("\n--- SCRAPED DATA ACQUIRED ---")
print(json.dumps(data, indent=4, ensure_ascii=False))

Launching headless browser to scrape: https://quotes.toscrape.com/...
Found 10 quotes. Extracting data...

--- SCRAPED DATA ACQUIRED ---
[
    {
        "quote": "The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.",
        "author": "Albert Einstein",
        "tags": [
            "change",
            "deep-thoughts",
            "thinking",
            "world"
        ]
    },
    {
        "quote": "It is our choices, Harry, that show what we truly are, far more than our abilities.",
        "author": "J.K. Rowling",
        "tags": [
            "abilities",
            "choices"
        ]
    },
    {
        "quote": "There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.",
        "author": "Albert Einstein",
        "tags": [
            "inspirational",
            "life",
            "live",
            "miracle",
            "miracles"
  